# bsc_03 — Headroom của NGỮ CẢNH BỀ MẶT (trước khi xây graph)

Stage 1 thất bại với cơ chế: **ảo giác ở vùng `absent` chiếm ~82% khối lượng lỗi vùng mỏng**;
recall 81% (sụn dày) → 37% (sụn mỏng). Xem `STAGE1_CONCLUSION.md`.

**Giả thuyết sửa:** cho mỗi node thấy **láng giềng trên bề mặt xương** (plan §4.4 Step 5 —
intra-surface message passing) để biết *"cả vùng này không có sụn"*. Tia 1D vứt bỏ ngữ cảnh
tiếp tuyến; lưới trên bề mặt trả lại ngữ cảnh ở thang **cm** — đúng thang mà vùng mất sụn
tồn tại.

**Nhưng plan cảnh báo ngược lại** (§4.4 Step 5 + test MM3): làm mượt trên bề mặt có thể
**lấp luôn focal defect thật**. Với failure mode là ảo giác, nó đi hai chiều ngược nhau:

| | |
|---|---|
| FP **rời rạc** | hàng xóm bỏ phiếu **dập được** → graph hữu ích |
| FP thành **mảng** | hàng xóm **củng cố** cái sai → graph làm tệ hơn |

⇒ **Đo trước, xây sau** — đúng tinh thần M0 (ROI cascade mất một chu kỳ vì xây trước khi đo).

## Bốn phép đo

1. **Coherence** — vắng sụn có liên tục theo không gian không? *(có tín hiệu để khai thác?)*
2. **Neighbor-oracle** — nếu **biết presence GT của láng giềng**, đoán được presence của node
   không? ⇒ **trần** của ngữ cảnh bề mặt.
3. **FP isolation** — FP của model rời rạc hay thành mảng? ⇒ graph giúp hay hại.
4. **Smoothing sweep** — "graph nhà nghèo": làm mượt presence rồi đo lại lỗi vùng mỏng.
   **Không train gì**, dùng checkpoint đã có.

## Cổng quyết định — ghi TRƯỚC khi chạy

| Tiêu chí | Ngưỡng |
|---|---|
| `signal_exists` — coherence lift | > 1.5 |
| `fp_isolated` — % láng giềng đoán đúng quanh FP | > 0.60 |
| `smoothing_helps` — giảm lỗi vùng mỏng | > 0.10mm |

**Cả ba đạt ⇒ CÓ HEADROOM**, đáng xây neighbor-pooling/graph.
**Không đủ ⇒ ngữ cảnh bề mặt không cứu được failure mode này** — thêm một bằng chứng cho
kết luận âm tính, và tiết kiệm cả Stage 2.

### 0. Config

In [1]:
from google.colab import drive
drive.mount("/content/drive")

REPO_URL, REPO_DIR = "https://github.com/AIVIETNAM-AIO-Tuan/bsCart-net.git", "/content/repo"
import os, sys, glob, json
if not os.path.isdir(f"{REPO_DIR}/bsc"):
    !git clone -q $REPO_URL $REPO_DIR
!cd $REPO_DIR && git pull -q
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
!pip install -q nibabel SimpleITK 2>/dev/null

for _m in [k for k in list(sys.modules) if k == "bsc" or k.startswith("bsc.")]:
    del sys.modules[_m]

import numpy as np
from tqdm.auto import tqdm
from bsc import core, metrics, headroom, io_utils, model as M, mvp
from bsc import atlas as atlas_mod, experiment as X, surface_context as SC
from bsc.core import RayConfig

# TU TIM duong dan thay vi hardcode - Drive hay bi sap xep lai (nnUNet_raw da tung
# nam o MyDrive/, roi chuyen vao OAI_seg/). Hardcode lam cell no IndexError kho hieu.
D = "/content/drive/MyDrive"

def _find_dir(name, hints=()):
    for h in hints:
        if os.path.isdir(h):
            return h
    for pat in (f"{D}/{name}", f"{D}/*/{name}", f"{D}/*/*/{name}"):
        for d in sorted(glob.glob(pat)):
            if os.path.isdir(d):
                return d
    return None

RAW_ROOT = _find_dir("nnUNet_raw", [f"{D}/OAI_seg/nnUNet_raw", f"{D}/nnUNet_raw"])
BSC_ROOT = _find_dir("bsc", [f"{D}/OAI_seg/bsc", f"{D}/bsc"])
assert RAW_ROOT, f"Khong tim thay nnUNet_raw duoi {D} (do sau <=2). Kiem lai Drive."
assert BSC_ROOT, f"Khong tim thay thu muc bsc duoi {D} (do sau <=2)."
RAW = f"{RAW_ROOT}/Dataset001_KneeOA"
print("RAW      =", RAW)
print("BSC_ROOT =", BSC_ROOT)

cfg = RayConfig()
CART = {"femoral_cart": 2, "med_tib_cart": 4}
BONE = {"femoral_cart": 1, "med_tib_cart": 3}
CLS = "femoral_cart"

_labs = sorted(glob.glob(f"{RAW}/labelsTr/oaizib_*.nii.gz"))
assert _labs, f"Khong co nhan nao o {RAW}/labelsTr - kiem lai duong dan."
SP = tuple(float(x) for x in io_utils.load_nii(_labs[0])[1])
print(f"{len(_labs)} nhan | spacing {tuple(round(x, 4) for x in SP)}"
      f" | SC san sang: {hasattr(SC, 'surface_context_case')}")


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 48.4 MB/s eta 0:00:00
RAW      = /content/drive/MyDrive/OAI_seg/nnUNet_raw/Dataset001_KneeOA
BSC_ROOT = /content/drive/MyDrive/bsc
404 nhan | spacing (0.3646, 0.3646, 0.7) | SC san sang: True


### 1. Nạp split + atlas + **checkpoint đã khóa**

Dùng lại canonical P2 (`4e133ba5fa0a74c6`) — **không train gì mới**. Nếu chưa có checkpoint
(session mới, chưa chạy Phase A) thì cell tự train lại P2 config cũ.

In [2]:
SPLITS = f"{BSC_ROOT}/splits/splits_zib_v1_fixed.json"
N_TRAIN, N_VAL = 40, 10

cases_all = sorted(os.path.basename(p)[:-len("_0000.nii.gz")]
                   for p in glob.glob(f"{RAW}/imagesTr/*_0000.nii.gz"))
cases_all = [c for c in cases_all if c.startswith("oaizib_")]
fold_of = json.load(open(SPLITS))["fold_of"]
zib = [c for c in cases_all if c in fold_of]
train_ids = [c for c in zib if fold_of[c] != 0][:N_TRAIN]
val_ids   = [c for c in zib if fold_of[c] == 0][:N_VAL]

CV_DIR = glob.glob(f"{BSC_ROOT}/baselines/ds020/**/nnUNetTrainer_150epochs*/", recursive=True)[0]
def baseline_path(cid):
    p = glob.glob(f"{CV_DIR}/fold_*/validation/{cid}.nii.gz")
    return p[0] if p else None

SRC = mvp.NiftiCaseSource(RAW, CLS, CART[CLS], BONE[CLS], baseline_path)

# Atlas: NAP neu co, khong thi DUNG LAI (tat dinh - ra file y het). ~15 phut CPU.
ATLAS_PATH = f"{BSC_ROOT}/atlas/atlas_{CLS}_fold0.npz"
os.makedirs(os.path.dirname(ATLAS_PATH), exist_ok=True)
if os.path.exists(ATLAS_PATH):
    z = np.load(ATLAS_PATH, allow_pickle=True)
    ATLAS = atlas_mod.ArticularAtlas(z["prob"], int(z["n_bins"]), float(z["lo"]),
                                     float(z["hi"]), tuple(z["case_ids"]), 0, None)
    print(f"Nap atlas: {len(ATLAS.case_ids)} ca")
else:
    print(f"Chua co atlas -> dung lai tu {len(train_ids)} ca train (~15 phut CPU)")
    ATLAS = atlas_mod.build_articular_atlas(
        train_ids, lambda c: (SRC.mri(c), SRC.bone_gt(c), SRC.cart_gt(c)),
        SP, cfg, n_bins=24, min_count=3, fold=0)
    np.savez_compressed(ATLAS_PATH, prob=ATLAS.prob, n_bins=ATLAS.n_bins,
                        lo=ATLAS.lo, hi=ATLAS.hi, case_ids=np.array(ATLAS.case_ids))
    print(f"Da luu -> {ATLAS_PATH}")

atlas_mod.assert_no_leak(ATLAS, val_ids)
print(f"train {len(train_ids)} | val {len(val_ids)} | atlas {len(ATLAS.case_ids)} ca"
      f" | phu {np.isfinite(ATLAS.prob).mean():.1%} o luoi")

run = X.from_plan("P2", CLS, seed=1)
ckpt = sorted(glob.glob(f"{BSC_ROOT}/runs/{run.experiment_id}/*/model.pt"))
if ckpt:
    run_dir = os.path.dirname(ckpt[-1])
    net = mvp.load_run_model(run_dir, device="cuda")
    man = mvp.run_manifest(run_dir)
    print(f"Nap checkpoint {man['checkpoint_sha256']} | git {man['git_commit'][:8]}"
          f" | val occ-Dice {man['val_occ_dice']:.3f}")
else:
    print("Chua co checkpoint - train lai P2 config cu (~20 phut)")
    res = mvp.train_run(run, SRC, train_ids, val_ids, cfg, atlas=ATLAS,
                        rays_per_case=20000, epochs=30, device="cuda")
    run_dir = mvp.save_run(res, BSC_ROOT, cfg, epochs=30, lr=3e-4, rays_per_case=20000,
                           dataset_revision="Dataset001_KneeOA", git_cwd=REPO_DIR)
    net = mvp.load_run_model(run_dir, device="cuda")
    print("Da luu ->", run_dir)


Nap atlas: 40 ca
train 40 | val 10 | atlas 40 ca | phu 34.8% o luoi
Chua co checkpoint - train lai P2 config cu (~20 phut)
Da luu -> /content/drive/MyDrive/bsc/runs/MVP_FC_S0_D1_I2_H1_v1_Fold0_Seed1/b889b49a297a787c


### 2. Chạy 4 phép đo trên 10 ca val

Có checkpoint resume — đứt kết nối chạy lại chỉ làm ca còn thiếu.

In [3]:
K_NB = 8
ALPHAS = (0.0, 0.3, 0.5, 0.7, 1.0)
B0_MM = 0.5517                      # ResEnc baseline, loi bien vung mong
SC_CKPT = f"{run_dir}/surface_context_k{K_NB}.jsonl"

done = ({json.loads(l)["case"] for l in open(SC_CKPT)}
        if os.path.exists(SC_CKPT) else set())
rows = [json.loads(l) for l in open(SC_CKPT)] if done else []
print(f"Da co {len(done)} ca | con {len([c for c in val_ids if c not in done])}")

with open(SC_CKPT, "a") as fh:
    for cid in tqdm(val_ids, desc="surface context"):
        if cid in done:
            continue
        r = SC.surface_context_case(run, net, SRC, cid, cfg, ATLAS, k=K_NB,
                                    alphas=ALPHAS, device="cuda")
        if r:
            rows.append(r)
            fh.write(json.dumps(r, default=str) + chr(10)); fh.flush()

print(f"Xong {len(rows)} ca -> {SC_CKPT}")

Da co 0 ca | con 10


surface context:   0%|          | 0/10 [00:00<?, ?it/s]

Xong 10 ca -> /content/drive/MyDrive/bsc/runs/MVP_FC_S0_D1_I2_H1_v1_Fold0_Seed1/b889b49a297a787c/surface_context_k8.jsonl


### 3. Kết quả + cổng quyết định

In [4]:
s = SC.summarize_surface_context(rows, baseline_thin_mm=B0_MM)

print(f"n = {s['n']} ca val")
print()
print("[1] COHERENCE - vang sun co lien tuc theo khong gian?")
print(f"    node absent co {s['nb_absent_given_absent']:.1%} lang gieng cung absent"
      f"  (ty le nen {s['base_absent_rate']:.1%})")
print(f"    lift = {s['coherence_lift']:.2f}x   (cong >1.5)   -> {s['signal_exists']}")
print()
print("[2] NEIGHBOR-ORACLE - biet presence GT cua lang gieng thi doan duoc gi?")
print(f"    F1 {s['nb_oracle_f1']:.3f} | absent-recall {s['nb_oracle_absent_recall']:.3f}")
print("    Cao => ngu canh be mat DU de xac dinh presence (tran cao cho graph)")
print()
print("[3] FP ISOLATION - FP roi rac hay thanh mang?")
print(f"    {s['fp_nb_correct']:.1%} lang gieng quanh FP duoc doan DUNG   (cong >60%)"
      f"   -> {s['fp_isolated']}")
print("    >60% => FP roi rac, hang xom DAP duoc | <40% => FP thanh MANG, cung co cai sai")
print()
print("[4] SMOOTHING SWEEP - lam muot presence tren be mat (graph nha ngheo)")
print(f"    {'alpha':>7}{'thin err':>11}{'vs B0':>9}{'presF1':>9}{'absentRec':>11}")
for a, v in s["sweep"].items():
    e = v["thin_err_mm"]
    mark = "  <- tot nhat" if a == s["best_alpha"] else ("  (xoa sach du doan)"
           if a in s["alphas_wiped_out"] else "")
    print(f"    {a:>7}{e:>10.3f}mm{e-B0_MM:>+9.3f}{v['presence_f1']:>9.3f}"
          f"{v['absent_recall']:>11.3f}{mark}")
print(f"    giam duoc {s['smoothing_gain_mm']:+.3f}mm  (cong >0.10)  -> {s['smoothing_helps']}")
print()
print("=" * 70)
print("=> " + s["verdict"])
print("=" * 70)

json.dump(s, open(f"{run_dir}/surface_context_summary.json", "w"), indent=2, default=str)
print(f"Da ghi {run_dir}/surface_context_summary.json")

n = 10 ca val

[1] COHERENCE - vang sun co lien tuc theo khong gian?
    node absent co 97.4% lang gieng cung absent  (ty le nen 36.7%)
    lift = 2.67x   (cong >1.5)   -> True

[2] NEIGHBOR-ORACLE - biet presence GT cua lang gieng thi doan duoc gi?
    F1 0.991 | absent-recall 0.983
    Cao => ngu canh be mat DU de xac dinh presence (tran cao cho graph)

[3] FP ISOLATION - FP roi rac hay thanh mang?
    38.1% lang gieng quanh FP duoc doan DUNG   (cong >60%)   -> False
    >60% => FP roi rac, hang xom DAP duoc | <40% => FP thanh MANG, cung co cai sai

[4] SMOOTHING SWEEP - lam muot presence tren be mat (graph nha ngheo)
      alpha   thin err    vs B0   presF1  absentRec
       0.00     1.752mm   +1.200    0.895      0.902
       0.30     1.713mm   +1.162    0.902      0.908
       0.50     1.696mm   +1.144    0.905      0.912
       0.70     1.684mm   +1.133    0.908      0.915  <- tot nhat
       1.00     1.686mm   +1.134    0.910      0.917
    giam duoc +0.067mm  (cong >0.10)  -> F

### 4. Đọc kết quả

**CÓ HEADROOM** (cả 3 cổng đạt) → xây neighbor-pooling: mỗi node gộp embedding của K láng
giềng trước khi vào presence head. Bản rẻ của §4.4 Step 5, chưa cần graph transformer.

**KHÔNG ĐỦ HEADROOM** → ngữ cảnh bề mặt không cứu được ảo giác absent. Ghép với các bằng
chứng đã có (loss/sampling ❌, dữ liệu 3.5× ❌), kết luận âm tính Stage 1 vững thêm — và
tiết kiệm được toàn bộ Stage 2.

**Trường hợp hỗn hợp** (ví dụ tín hiệu có nhưng FP thành mảng) là kết quả *có thông tin*:
nó nói ảo giác của model **có cấu trúc không gian**, tức model sai một cách nhất quán theo
vùng — gợi ý vấn đề nằm ở **tín hiệu ảnh tại vùng đó**, không phải ở thiếu ngữ cảnh.

⚠️ Mọi số ở đây đo trên **10 ca val** đã dùng nhiều lần cho chẩn đoán ⇒ là **engineering
diagnostic**, không phải final Gate. Nếu quyết định xây, phải đánh giá lại trên split phân
tầng (Phase C).

### 5. Ngữ cảnh THÔ có thêm thông tin không? (k-NN — quyết định slab/3D CNN)

**Đính chính phạm vi phép [1]–[4]:** chúng chỉ đo việc tổng hợp **dự đoán** của láng giềng
(làm mượt). Thất bại vì model đã sai theo mảng — *rác vào rác ra*. Chúng **không** trả lời:
*nếu cho model nhìn **đặc trưng MRI thô** của tia lân cận thì sao?*

Đó là thứ mà **3D CNN trên slab neo pháp tuyến** (plan §5.2 Extension B) làm, và nó khác hẳn
làm mượt đầu ra. Chính phép [2] (neighbor-oracle F1 **0.991**) gợi ý ngữ cảnh tiếp tuyến rất
giàu thông tin — chỉ là làm mượt dự đoán không chạm tới được.

**Phép đo:** k-NN phi tham số, **không train gì**. Học từ tia train, đo trên tia val, chỉ xét
**tia khó** (độ dày ≤1mm — nơi §3.7 thất bại).

| | Đặc trưng dùng |
|---|---|
| **A** | một tia (K×C) |
| **B** | tia + 4 láng giềng ghép lại |

k-NN là baseline mạnh nhất có thể (nó *thuộc lòng* toàn bộ tập train) — nếu nó không tách
được thì kiến trúc nào cũng khó.

**Cổng ghi TRƯỚC:**

| Tiêu chí | Ngưỡng | Ý nghĩa |
|---|---|---|
| `features_informative` | acc(A) > majority + 0.05 | đặc trưng tia **có** tách được absent/mỏng |
| `context_adds_info` | acc(B) − acc(A) > 0.03 | láng giềng **thô** thêm thông tin thật |

- **Cả hai đạt** ⇒ **slab + 3D CNN đáng xây** (đặc trưng có tín hiệu, ngữ cảnh bổ sung thêm).
- `features_informative` **False** ⇒ đặc trưng tia không phân biệt được absent vs sụn 0.3mm
  ⇒ **giới hạn thông tin**, không kiến trúc nào cứu — kể cả 3D CNN.
- `context_adds_info` **False** (nhưng A tốt) ⇒ thông tin nằm ở chính tia, láng giềng không
  thêm gì ⇒ slab không giúp, vấn đề ở tối ưu hóa/tham số hóa đầu ra.

In [ ]:
# 5. k-NN: dac trung MOT TIA vs TIA + LANG GIENG (khong train gi) ---------------
KNN = dict(k_nb=8, n_use=4, k_knn=15, rays_per_case=4000)
r5 = SC.raw_context_knn(run, SRC, train_ids[:20], val_ids, cfg, atlas=ATLAS,
                        hard_only=True, **KNN)

print(f"tia KHO (do day <=1mm): train {r5['n_train_rays']} | val {r5['n_val_rays']}")
print(f"ty le tia CO sun trong val: {r5['present_rate_val']:.1%}"
      f"  -> doan lop da so cho acc {r5['majority_baseline']:.3f}")
print()
print(f"{'dac trung':<22}{'acc':>8}{'F1':>8}{'absent-recall':>15}")
print(f"{'A: mot tia':<22}{r5['single_ray']['acc']:>8.3f}{r5['single_ray']['f1']:>8.3f}"
      f"{r5['single_ray']['absent_recall']:>15.3f}")
print(f"{'B: tia + 4 lang gieng':<22}{r5['with_neighbors']['acc']:>8.3f}"
      f"{r5['with_neighbors']['f1']:>8.3f}{r5['with_neighbors']['absent_recall']:>15.3f}")
print()
print(f"dac trung co tin hieu?   acc(A) - majority = {r5['single_ray']['acc']-r5['majority_baseline']:+.3f}"
      f"   (cong >+0.05)  -> {r5['features_informative']}")
print(f"lang gieng them thong tin? acc(B) - acc(A)  = {r5['gain_acc']:+.3f}"
      f"   (cong >+0.03)  -> {r5['context_adds_info']}")
print()
if r5["features_informative"] and r5["context_adds_info"]:
    v = "SLAB + 3D CNN DANG XAY - dac trung co tin hieu VA lang gieng them thong tin"
elif not r5["features_informative"]:
    v = "GIOI HAN THONG TIN - dac trung tia khong tach duoc absent vs mong. 3D CNN cung khong cuu."
else:
    v = "LANG GIENG KHONG THEM GI - slab khong giup; van de o toi uu hoa/tham so hoa dau ra"
print("=" * 78)
print("=> " + v)
print("=" * 78)

r5["verdict"] = v
json.dump(r5, open(f"{run_dir}/raw_context_knn.json", "w"), indent=2, default=str)
print(f"Da ghi {run_dir}/raw_context_knn.json")